## Run the arbitrage processing notebook

Make sure these services are running:

- **Kafka** 
- **NiFi** (so it’s pushing to Kafka topics)

Also check you have data flowing into topics:

- `crypto.prices.binance` (run the following command to confirm: `kafka-console-consumer.sh   --bootstrap-server localhost:9092   --topic crypto.prices.binance`)
- `crypto.prices.coinbase` (run the following command to confirm: `kafka-console-consumer.sh   --bootstrap-server localhost:9092   --topic crypto.prices.coinbase`)


## Environment

In [ ]:
import os
# Kafka connector jars for Spark Structured Streaming
# IMPORTANT: Run this cell BEFORE creating SparkSession (and ideally before importing pyspark).
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.4,org.apache.spark:spark-token-provider-kafka-0-10_2.12:3.5.4 pyspark-shell'


Initilize the Spark session, named "Arbitrage opportunity finder"

In [ ]:
from pyspark.sql.session import SparkSession

spark_session = \
  SparkSession.builder\
              .appName("Arbitrage opportunity finder")\
              .getOrCreate()

print(f"This cluster relies on Spark '{spark_session.version}'")

OpenJDK 64-Bit Server VM warning: Unable to get SVE vector length on this system. Disabling SVE. Specify -XX:UseSVE=0 to shun this warning.
OpenJDK 64-Bit Server VM warning: Unable to get SVE vector length on this system. Disabling SVE. Specify -XX:UseSVE=0 to shun this warning.
26/01/11 20:29:29 WARN Utils: Your hostname, osbdet resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s1)
26/01/11 20:29:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/osbdet/.ivy2/cache
The jars for the packages stored in: /home/osbdet/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.spark#spark-token-provider-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bf209663-4029-40e5-b193-22db95911853;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/osbdet/.jupyter_venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.4 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.4 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 204ms :: artifacts dl 25ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#

This cluster relies on Spark '3.5.4'


## Start Nifi and Kafka

1. Run Nifi Streaming Flow for Binance and Coinbase
2. Start two Kafka topics

   `kafka-console-consumer.sh   --bootstrap-server localhost:9092   --topic crypto.prices.binance`

   `kafka-console-consumer.sh   --bootstrap-server localhost:9092   --topic crypto.prices.coinbase`

## Read Coinbase Data From Kafka

Read real-time data from the `crypto.prices.coinbase` Kafka topic. Define a schema, parse the incoming JSON, and perform data cleaning and unification, including handling various field names, normalizing timestamp formats, extracting base/quote currencies, casting data types, and filtering out unwanted records.

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType

# Read Coinbase Kafka messages
coinbase_raw = (
    spark_session.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "localhost:9092")
      .option("subscribe", "crypto.prices.coinbase")
      .option("startingOffsets", "latest")
      .load()
      .select(F.col("value").cast("string").alias("json_str"))
)

# Coinbase payload (expected). If NiFi standardizes Coinbase to the same format as Binance,
# this schema still works because we use robust extraction below.
coinbase_schema = StructType([
    StructField("symbol", StringType(), True),        # e.g. BTC-USD
    StructField("bid", StringType(), True),
    StructField("ask", StringType(), True),
    StructField("bidPrice", StringType(), True),      # fallback if using Binance-like keys
    StructField("askPrice", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("ts", StringType(), True),
    StructField("time", StringType(), True),
    StructField("timestamp", StringType(), True),
])

# Parse JSON
coinbase_parsed0 = (
    coinbase_raw
      .select(from_json(col("json_str"), coinbase_schema).alias("d"))
      .select("d.*")
)

# Choose bid/ask fields robustly (bid/ask OR bidPrice/askPrice)
coinbase_bid = F.coalesce(col("bid"), col("bidPrice"))
coinbase_ask = F.coalesce(col("ask"), col("askPrice"))

# Choose timestamp field robustly
coinbase_event = F.coalesce(col("event_time"), col("ts"), col("time"), col("timestamp"))

# Normalize timezone +01:00 -> +0100 so Spark parses reliably
coinbase_event_norm = F.regexp_replace(coinbase_event, r"([+-]\d\d):(\d\d)$", r"\1\2")

coinbase_unified = (
    coinbase_parsed0
      .withColumn("exchange", F.lit("coinbase")) # Adds a new column 'exchange' and sets it to 'coinbase'
      .withColumn("symbol", col("symbol")) # Checks that the 'symbol' column is present
      .withColumn("base", F.split(col("symbol"), "-").getItem(0)) # Extracts the base currency (e.g., 'BTC' from 'BTC-USD')
      .withColumn("quote", F.split(col("symbol"), "-").getItem(1)) # Extracts the quote currency (e.g., 'USD' from 'BTC-USD')
      .withColumn("bid", coinbase_bid.cast("double")) # Casts bid price to double
      .withColumn("ask", coinbase_ask.cast("double")) # Casts ask price to double
      .withColumn("event_ts", F.to_timestamp(coinbase_event_norm)) # Converts normalized timestamp string to proper timestamp type
      .select("exchange", "base", "quote", "bid", "ask", "event_ts") # Selects and orders the final set of columns for a unified schema
      .filter(col("event_ts").isNotNull() & (col("base") != "")) # Filter out rows where timestamp is null or base is empty
      .filter(col("bid").isNotNull() & col("ask").isNotNull()) # Filter out rows where bid or ask price is null
      .dropDuplicates(["exchange", "base", "quote", "event_ts"]) # Exactly once processing by dropping duplicates based on these columns within the stream
)


## Read Binance Data from Kafka

Similar to the Coinbase cell, we read and prepare Binance data from the `crypto.prices.binance` Kafka topic.

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType

binance_raw = (
    spark_session.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "localhost:9092")
      .option("subscribe", "crypto.prices.binance")
      .option("startingOffsets", "latest")
      .load()
      .select(F.col("value").cast("string").alias("json_str"))
)

# Binance payload produced by NiFi (based on your sample)
binance_schema = StructType([
    StructField("symbol", StringType(), True),      # e.g. BTCUSDT
    StructField("bidPrice", StringType(), True),
    StructField("bidQty", StringType(), True),
    StructField("askPrice", StringType(), True),
    StructField("askQty", StringType(), True),
    StructField("event_time", StringType(), True),  # should be quoted string; we fix if not
])

# Fix invalid JSON if NiFi accidentally emits unquoted event_time:
#  "event_time":2025-...+01:00  ->  "event_time":"2025-...+01:00"
binance_fixed = binance_raw.withColumn(
    "json_fixed",
    F.regexp_replace(
        col("json_str"),
        r'"event_time"\s*:\s*([0-9]{4}-[0-9]{2}-[0-9]{2}T[^",}]+)',
        r'"event_time":"\1"'
    )
)

binance_parsed = (
    binance_fixed
      .select(from_json(col("json_fixed"), binance_schema).alias("d"))
      .select("d.*")
)

# Normalize timezone +01:00 -> +0100 for reliable parsing
binance_event_norm = F.regexp_replace(col("event_time"), r"([+-]\d\d):(\d\d)$", r"\1\2")

binance_unified = (
    binance_parsed
      .withColumn("exchange", F.lit("binance")) # Add the 'exchange' column and set it to 'binance'
      # assumes all your binance pairs are *USDT; adjust if you use others
      .withColumn("base", F.regexp_extract(col("symbol"), r"^(.+?)USDT$", 1)) # Extracts the base currency (e.g., 'BTC' from 'BTCUSDT')
      .withColumn("quote", F.lit("USDT"))      # Sets the quote currency to USDT
      .withColumn("bid", col("bidPrice").cast("double")) # Casts bidPrice to double
      .withColumn("ask", col("askPrice").cast("double")) # Casts askPrice to double
      .withColumn("event_ts", F.to_timestamp(binance_event_norm)) # Converts normalized timestamp to proper timestamp type
      .select("exchange", "base", "quote", "bid", "ask", "event_ts") # Selects and orders the final set of columns for a unified schema
      .filter(col("event_ts").isNotNull() & (col("base") != "")) # Filters out rows where event_ts is null or base is empty
      .filter(col("bid").isNotNull() & col("ask").isNotNull()) # Filters out rows where bid or ask price is null
      .dropDuplicates(["exchange", "base", "quote", "event_ts"]) # Exactly once processing by dropping duplicates based on these columns within the stream
)


This cell merges Binance and Coinbase price streams, aggregates ticks into 1-second snapshots, and computes cross-exchange arbitrage opportunities.

For each asset and second, it keeps the latest bid/ask from each exchange, aligns them side-by-side, and calculates the best arbitrage direction (buy on one exchange, sell on the other) along with the resulting spread.

The output is a clean, de-duplicated per-second arbitrage signal ready to be sent downstream (e.g. Kafka, alerts, or strategy logic).

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col

# --- Config ---
WATERMARK = "10 seconds"          # tolerate late data
WINDOW = "1 second"               # compute opportunities each second

# Combine both exchanges into one stream
ticks = binance_unified.unionByName(coinbase_unified)

# 1) Snapshot per second per (base, exchange) using the latest tick in that second
per_exch_1s = (
    ticks
      .withWatermark("event_ts", WATERMARK)
      .groupBy(
          F.window(col("event_ts"), WINDOW).alias("w"),
          col("base").alias("asset"),
          col("exchange")
      )
      .agg(
          F.max_by(col("bid"), col("event_ts")).alias("bid"),
          F.max_by(col("ask"), col("event_ts")).alias("ask"),
          F.max(col("event_ts")).alias("last_event_ts")
      )
)

# 2) Pivot so each row has both exchanges side-by-side (binance_* and coinbase_*)
pivoted = (
    per_exch_1s
      .groupBy("w", "asset")
      .pivot("exchange", ["binance", "coinbase"])
      .agg(
          F.first("bid").alias("bid"),
          F.first("ask").alias("ask")
      )
)

# After pivot, columns look like: binance_bid, binance_ask, coinbase_bid, coinbase_ask
arb_simple = (
    pivoted
      .withColumnRenamed("binance_bid", "binance_bid")
      .withColumnRenamed("binance_ask", "binance_ask")
      .withColumnRenamed("coinbase_bid", "coinbase_bid")
      .withColumnRenamed("coinbase_ask", "coinbase_ask")
      .filter(col("binance_bid").isNotNull() & col("binance_ask").isNotNull()
              & col("coinbase_bid").isNotNull() & col("coinbase_ask").isNotNull())
      .withColumn("ts", col("w.end"))
      # Spread = sell_bid - buy_ask
      .withColumn("spread_buy_binance_sell_coinbase", col("coinbase_bid") - col("binance_ask"))
      .withColumn("spread_buy_coinbase_sell_binance", col("binance_bid") - col("coinbase_ask"))
      .withColumn(
          "best_spread",
          F.greatest(col("spread_buy_binance_sell_coinbase"), col("spread_buy_coinbase_sell_binance"))
      )
      .withColumn(
          "direction",
          F.when(col("spread_buy_binance_sell_coinbase") >= col("spread_buy_coinbase_sell_binance"),
                 F.lit("BUY_BINANCE_SELL_COINBASE"))
           .otherwise(F.lit("BUY_COINBASE_SELL_BINANCE"))
      )
      .select(
          "ts", "asset", "direction", "best_spread",
          "spread_buy_binance_sell_coinbase",
          "spread_buy_coinbase_sell_binance",
          "binance_bid", "binance_ask",
          "coinbase_bid", "coinbase_ask"
      )
)

# Optional: console debug
# arb_simple.writeStream.format("console").outputMode("append").option("truncate","false").start()


In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col

# Output Detected Arbitrage Signals to Kafka

1. Convert each row in the **arb_simple** DataFrame into a single JSON string
2. Rename the JSON string column to `value` which is what Kafka expects for message payloads
3. Use `writeStream` to continuously write the formatted data to the Kafka topic `crypto.arb.signals`
4. Specify `checkpointLocation` to ensure exactly-once processing. If the application crashes, it can restart from the checkpoint and avoid data loss or reprocessing
5. Use `outputMode("append")` so only new records appended to the streaming DataFrame since the last trigger will be written to Kafka
6. `start()` the stream

In [ ]:
arb_out = arb_simple.select(
    F.to_json(F.struct(
        col("ts"),
        col("asset"),
        col("direction"),
        col("best_spread"),
        col("spread_buy_binance_sell_coinbase"),
        col("spread_buy_coinbase_sell_binance"),
        col("binance_bid"), col("binance_ask"),
        col("coinbase_bid"), col("coinbase_ask"),
    )).alias("value")
)

kafka_q = (
    arb_out.writeStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "localhost:9092")
      .option("topic", "crypto.arb.signals")
      .option("checkpointLocation", "/tmp/spark-checkpoints/crypto-arb-signals-min-7")
      .outputMode("append")
      .start()
)


26/01/11 20:29:32 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/01/11 20:29:32 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/01/11 20:29:32 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/01/11 20:29:45 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 58:=======================================>              (148 + 2) / 200]